In [6]:
!pip install langchain-google-genai langchain-huggingface langchain-community pypdf langgraph faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 22.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found exist

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langgraph.graph import StateGraph, END, START
import os
import requests
from typing import TypedDict, List

/tmp/ipykernel_2114/1537478065.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
print(os.getcwd())
from google.colab import files

uploaded = files.upload()

/content


Saving Introduction_to_RAG.pdf to Introduction_to_RAG.pdf


In [ ]:
os.environ['OPENAI_API_KEY'] = "API_KEY"
openai_api_key = os.getenv('OPENAI_API_KEY')
MODEL = "gemini-2.5-flash"

data = PyPDFLoader("./Introduction_to_RAG.pdf").load()

class State(TypedDict):
  retrieved_docs: str
  response: str
  query: str

In [9]:
llm_chat = ChatGoogleGenerativeAI(temperature=0, api_key=openai_api_key, model=MODEL)
emb_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(documents=data, embedding=emb_model)
retriever = vector_store.as_retriever()
query = "What is the difference between LangChain and LangGraph"

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
def retriever_node(state:State):
  docs = retriever.invoke(state['query'])
  docs_retrieved = [d.page_content for d in docs][0]
  return {'retrieved_docs': docs_retrieved}

def response_node(state:State):
  prompt = f"use the context {state['retrieved_docs']} to answer the following query {state['query']}"
  generation = llm_chat.invoke(prompt).content
  return {'response': generation}

In [16]:
graph = StateGraph(State)

graph.add_node("Retriever", retriever_node)
graph.add_node("Response", response_node)

graph.add_edge(START, "Retriever")
graph.add_edge("Retriever", "Response")
graph.add_edge("Response", END)

app = graph.compile()

In [17]:
answer = app.invoke({ 'query': query })
print(answer["response"])

The context you provided perfectly illustrates the problem that LangChain and LangGraph aim to solve, especially in the realm of Retrieval-Augmented Generation (RAG).

Let's break down the difference between LangChain and LangGraph, using your context:

---

**The Problem (as per your context):**

*   Traditional LLMs, like the GPT-2 example you gave, rely *solely on pre-trained data*.
*   This data can become **outdated or incomplete**, leading to **hallucinations** (like inventing "Dr. Emily Zhang" for the 2023 Nobel Prize).
*   **Retraining is expensive** and resource-intensive.
*   **RAG is the solution:** It acts like a "bridge" to connect the LLM to external, up-to-date knowledge (documents, online search, knowledge bases) to fetch relevant content and generate accurate, grounded responses.

---

**Where LangChain and LangGraph come in:**

Both LangChain and LangGraph are frameworks designed to help developers build sophisticated applications *around* Large Language Models, often